In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ArbresRemarquables").getOrCreate()
sc = spark.sparkContext  # ← obligatoire pour utiliser sc.parallelize()

# Données fictives
data = [
    {"nom": "Chêne",   "annee": "1800", "hauteur": "15", "arrond": "75001"},
    {"nom": "Tilleul", "annee": "Inconnue", "hauteur": "10", "arrond": "75002"},
    {"nom": "Platane", "annee": "1950", "hauteur": None,  "arrond": "75001"},
    {"nom": "Hêtre",   "annee": "1600", "hauteur": "20", "arrond": "75003"},
    {"nom": "Sapin",   "annee": None,   "hauteur": "8",  "arrond": "75002"},
]

rdd_test = sc.parallelize(data)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 10:18:38 WARN Utils: Your hostname, PYTHON-08, resolves to a loopback address: 127.0.1.1; using 172.22.114.75 instead (on interface eno1)
26/04/17 10:18:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 10:18:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# ── ÉTAPE 2 : filter ────────────────────────────────────────────────────────
# On garde uniquement les lignes avec une année valide
def annee_valide(row):
    val = row["annee"]
    if val is None:
        return False
    try:
        int(val)
        return True
    except (ValueError, TypeError):
        return False

rdd_filtre = rdd_test.filter(annee_valide)

print("=== Après filter (annee valide) ===")
for row in rdd_filtre.collect():
    print(row)
# → Tilleul et Sapin ont disparu

=== Après filter (annee valide) ===


{'nom': 'Chêne', 'annee': '1800', 'hauteur': '15', 'arrond': '75001'}
{'nom': 'Platane', 'annee': '1950', 'hauteur': None, 'arrond': '75001'}
{'nom': 'Hêtre', 'annee': '1600', 'hauteur': '20', 'arrond': '75003'}


In [4]:
# ── ÉTAPE 3 : map ───────────────────────────────────────────────────────────
# On transforme chaque ligne en un simple âge
rdd_ages = rdd_filtre.map(lambda row: 2026 - int(row["annee"]))

print("=== Après map (calcul âge) ===")
print(rdd_ages.collect())

=== Après map (calcul âge) ===
[226, 76, 426]


In [5]:
# ── ÉTAPE 4 : reduce ────────────────────────────────────────────────────────
# On garde le maximum
age_max = rdd_ages.reduce(lambda a, b: a if a > b else b)

print("=== Après reduce (max) ===")
print(f"Âge max : {age_max} ans")

=== Après reduce (max) ===
Âge max : 426 ans


In [7]:
# ── ÉTAPE 5 : map en tuple + reduceByKey ────────────────────────────────────
# Pour compter les arbres par arrondissement
rdd_arrond = rdd_test.map(lambda row: (row["arrond"], 1))

print("=== Après map en tuple ===")
print(rdd_arrond.collect())

rdd_compte = rdd_arrond.reduceByKey(lambda a, b: a + b)

print("=== Après reduceByKey ===")
print(rdd_compte.collect())


=== Après map en tuple ===
[('75001', 1), ('75002', 1), ('75001', 1), ('75003', 1), ('75002', 1)]
=== Après reduceByKey ===
[('75001', 2), ('75003', 1), ('75002', 2)]


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = SparkSession.builder.appName("Taxi").getOrCreate()
sc = spark.sparkContext
df = spark.read.parquet("yellow_tripdata_2026-01.parquet")
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [22]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [25]:
print(f"le nombre totale de lignes: {df.count()}")

le nombre totale de lignes: 3724889


In [17]:
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in df.columns]).show(vertical=True)

-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 1088058 
 trip_distance         | 0       
 RatecodeID            | 1088058 
 store_and_fwd_flag    | 1088058 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 1088058 
 Airport_fee           | 1088058 
 cbd_congestion_fee    | 0       



In [41]:
df.groupBy("VendorID").count().show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 710425|
|       7|  44705|
|       2|2965742|
|       6|   4017|
+--------+-------+



In [58]:
from pyspark.sql.functions import col

df.filter(col("passenger_count").isNull()).groupBy("VendorID").count().show()

+--------+------+
|VendorID| count|
+--------+------+
|       1|139406|
|       6|  4017|
|       2|944635|
+--------+------+



In [59]:
df_clean = df.fillna({
    "passenger_count" : 1,
    "RatecodeID": 1,               
    "store_and_fwd_flag": "N",    
    "congestion_surcharge": 0.0,   
    "Airport_fee": 0.0

})
print(f"Nulls restants après fillna : {df_clean.filter(col('passenger_count').isNull()).count()}")

Nulls restants après fillna : 0


In [60]:


df_clean.filter(col("passenger_count").isNull()).groupBy("VendorID").count().show()

+--------+-----+
|VendorID|count|
+--------+-----+
+--------+-----+



In [64]:
print(f"Avant filtre : {df_clean.count()} lignes")

df_clean = df_clean.filter(
    (col("fare_amount") > 0) & (col("fare_amount") <= 500) &
    (col("trip_distance") > 0) & (col("trip_distance") <= 200)
)

print(f"Après filtre aberrants : {df_clean.count()} lignes")

Avant filtre : 3560706 lignes
Après filtre aberrants : 3560706 lignes


In [62]:
#df_clean = df_clean.filter(
#    (col("fare_amount") <= 0) & (col("fare_amount") > 500) &
#    (col("trip_distance") == 0) & (col("trip_distance") > 200)
#)

#print(f"Après filtre aberrants : {df_clean.count()} lignes")

In [66]:
total = df.count()
conserve = df_clean.count()
print(f"Total initial     : {total} lignes")
print(f"Total conservé    : {conserve} lignes")
print(f"Lignes conservées : {conserve / total * 100:}%")

Total initial     : 3724889 lignes
Total conservé    : 3560706 lignes
Lignes conservées : 95.5922713401661%


In [69]:
# La plus chère
df_clean.orderBy(col("fare_amount").desc()).select(
    "fare_amount", "trip_distance", "tpep_pickup_datetime"
).show(1)

# La plus longue
df_clean.orderBy(col("trip_distance").desc()).select(
    "fare_amount", "trip_distance", "tpep_pickup_datetime"
).show(1)

+-----------+-------------+--------------------+
|fare_amount|trip_distance|tpep_pickup_datetime|
+-----------+-------------+--------------------+
|      500.0|         18.8| 2026-01-18 18:48:03|
+-----------+-------------+--------------------+
only showing top 1 row
+-----------+-------------+--------------------+
|fare_amount|trip_distance|tpep_pickup_datetime|
+-----------+-------------+--------------------+
|       33.5|        192.8| 2026-01-08 11:25:05|
+-----------+-------------+--------------------+
only showing top 1 row


# Kit iteration 2

### 1 – Lire le Parquet et mesurer le temps

In [71]:
import time

start = time.time()
df = spark.read.parquet("yellow_tripdata_2026-01.parquet")
df.count()  # force la lecture complète
end = time.time()

print(f"Temps lecture Parquet : {end - start:.2f} secondes")
print(f"Nombre de lignes : {df.count()}")

Temps lecture Parquet : 0.20 secondes
Nombre de lignes : 3724889


### 2 – Convertir en CSV

In [76]:
df.write.mode("overwrite").option("header", True).csv("yellow_tripdata_2026-01_CSV")
print("Export CSV terminé")

Export CSV terminé


### 3 – Lire le CSV et comparer

In [77]:
start = time.time()
df_csv = spark.read.option("header", True).option("inferSchema", True).csv("yellow_tripdata_2026-01_CSV")
df_csv.count()
end = time.time()

print(f"Temps lecture CSV : {end - start:.2f} secondes")

Temps lecture CSV : 3.95 secondes


### 4 – Projection : lire uniquement 3 colonnes

In [78]:
colonnes = ["fare_amount", "trip_distance", "total_amount"]

# Depuis Parquet
start = time.time()
df_parquet_proj = spark.read.parquet("yellow_tripdata_2026-01.parquet").select(colonnes)
df_parquet_proj.count()
end = time.time()
print(f"Parquet 3 colonnes : {end - start:.2f} secondes")

# Depuis CSV
start = time.time()
df_csv_proj = spark.read.option("header", True).option("inferSchema", True).csv("yellow_tripdata_2026-01_CSV").select(colonnes)
df_csv_proj.count()
end = time.time()
print(f"CSV 3 colonnes : {end - start:.2f} secondes")

Parquet 3 colonnes : 0.21 secondes


CSV 3 colonnes : 3.91 secondes


### Pandas vs Spark pour lire CSV

In [91]:
import pandas as pd
import time

# Pandas - Parquet
start = time.time()
df_pd = pd.read_parquet("yellow_tripdata_2026-01.parquet")
end = time.time()
print(f"Pandas Parquet : {end - start:.2f} secondes")

# df_pd.to_csv("yellow_tripdata_2026-01.csv")
# Pandas - CSV (après export)
start = time.time()
# df_pd_csv = pd.read_csv("yellow_tripdata_2026-01_CSV/part-00000-b341c75e-504a-4498-855e-4c8d798370ed-c000.csv")  # un seul fichier
df_pd_csv = pd.read_csv("yellow_tripdata_2026-01.csv")
end = time.time()
print(f"Pandas CSV : {end - start:.2f} secondes")

Pandas Parquet : 0.31 secondes


/tmp/ipykernel_54118/622122278.py:14: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pd_csv = pd.read_csv("yellow_tripdata_2026-01.csv")


Pandas CSV : 9.24 secondes
